# ArabCupVision — réentraînement sur GPU (Google Colab)

Ce notebook relance `training/train_all.py` avec des réglages GPU (plus d'epochs, images plus grandes).

1. *Exécution > Modifier le type d'exécution > GPU*.
2. Ajoutez la clé `ROBOFLOW_API_KEY` dans les **secrets Colab** (icône clé).
3. Exécutez les cellules dans l'ordre, puis téléchargez `models/detector.pt` et `models/pitch_keypoints.pt` dans le dossier `models/` du projet.

In [ ]:
# Récupération du projet : clonez votre dépôt (URL à adapter) ou importez une archive du projet.
REPO_URL = ""  # ex. https://<jeton>@github.com/<compte>/arabcupvision.git
import os
if REPO_URL and not os.path.exists("arabcupvision"):
    !git clone -q $REPO_URL arabcupvision
%cd arabcupvision

In [ ]:
!pip -q install "ultralytics>=8.3" "roboflow>=1.1" pyyaml

In [ ]:
import os
from google.colab import userdata
os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")

In [ ]:
import yaml
cfg = yaml.safe_load(open("training/train.yaml", encoding="utf-8"))
cfg["device"] = 0
cfg["workers"] = 2
for name, job in cfg["jobs"].items():
    job["epochs"] = 100 if name == "detector" else 200
    job["imgsz"] = 1280 if name == "detector" else 640
    job["batch"] = 8 if name == "detector" else 16
    job["base_model"] = "yolov8s.pt" if name == "detector" else "yolov8s-pose.pt"
    job["time_hours"] = None
yaml.safe_dump(cfg, open("training/train_gpu.yaml", "w", encoding="utf-8"), sort_keys=False)
print(yaml.safe_dump(cfg, sort_keys=False))

In [ ]:
!python training/train_all.py --config training/train_gpu.yaml

In [ ]:
from google.colab import files
files.download("models/detector.pt")
files.download("models/pitch_keypoints.pt")